In [ ]:
%pip install qiskit-nature pyscf
%pip install matplotlib
import qiskit
import qiskit_ibm_runtime

print("Qiskit:",qiskit.__version__)
print("Qiskit IBM Runtime:",qiskit_ibm_runtime.__version__)

In [1]:
import numpy as np

from qiskit.quantum_info import SparsePauliOp

from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import ParityMapper


# ============================================================
# 1. MOLECULAR CONFIGURATION
# ============================================================

bond_distance=0.735
basis="sto3g"


# ============================================================
# 2. CREATE H2 MOLECULE
# ============================================================

driver=PySCFDriver(
    atom=f"H 0 0 0; H 0 0 {bond_distance}",
    basis=basis
)


# ============================================================
# 3. RUN PYSCF
# ============================================================

problem=driver.run()


# ============================================================
# 4. GET MOLECULAR HAMILTONIAN
# ============================================================

second_q_hamiltonian=problem.hamiltonian.second_q_op()


# ============================================================
# 5. GET NUMBER OF ELECTRONS
# ============================================================

num_particles=problem.num_particles

print("Number of particles:",num_particles)


# ============================================================
# 6. PARITY MAPPING
# ============================================================

mapper=ParityMapper(
    num_particles=num_particles
)

qubit_hamiltonian=mapper.map(
    second_q_hamiltonian
)


# ============================================================
# 7. CONVERT TO SPARSE PAULI OP
# ============================================================

hamiltonian=SparsePauliOp(
    qubit_hamiltonian.paulis,
    qubit_hamiltonian.coeffs
)


# ============================================================
# 8. PRINT HAMILTONIAN
# ============================================================

print("\n========================================")
print("H2 MOLECULAR HAMILTONIAN")
print("========================================")

print("Bond distance:",bond_distance,"Angstrom")
print("Basis:",basis)

print("\nHamiltonian:")
print(hamiltonian)

print("\nNumber of qubits:")
print(hamiltonian.num_qubits)

print("\nCoefficients:")

for pauli,coefficient in zip(
    hamiltonian.paulis,
    hamiltonian.coeffs
):

    print(
        str(pauli),
        ":",
        float(np.real(coefficient))
    )

: 

In [ ]:
# ============================================================
# HARDWARE VQE
# 2 QUBITS + 2 PARAMETERS + SPSA
# ============================================================

import numpy as np

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator
)

from qiskit_algorithms.optimizers import SPSA


# ============================================================
# 1. CHECK HAMILTONIAN
# ============================================================

print("\n========================================")
print("HARDWARE VQE")
print("========================================")

print("\nHamiltonian:")
print(hamiltonian)

print("\nNumber of qubits:",hamiltonian.num_qubits)

if hamiltonian.num_qubits != 2:

    raise ValueError(
        f"Hamiltonian has {hamiltonian.num_qubits} qubits. "
        "The VQE circuit below requires exactly 2 qubits."
    )


# ============================================================
# 2. EXACT REFERENCE ENERGY
# ============================================================

hamiltonian_matrix=hamiltonian.to_matrix()

eigenvalues=np.linalg.eigvalsh(
    hamiltonian_matrix
)

exact_energy=float(
    np.min(eigenvalues)
)

print("\nExact ground-state energy:")
print(exact_energy)


# ============================================================
# 3. CONNECT TO IBM QUANTUM
# ============================================================

import os
from qiskit_ibm_runtime import QiskitRuntimeService

os.environ["QISKIT_IBM_TOKEN"]="-______-"#API KEY

service=QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=os.environ["QISKIT_IBM_TOKEN"]
)

print("\nCONNECTING TO IBM QUANTUM...")

service=QiskitRuntimeService()

backend=service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=2
)

print("Backend:",backend.name)


# ============================================================
# 4. CREATE 2-QUBIT / 2-PARAMETER VQE CIRCUIT
# ============================================================

theta0=Parameter("theta0")
theta1=Parameter("theta1")

qc=QuantumCircuit(2)

qc.ry(theta0,0)
qc.ry(theta1,1)

qc.cx(0,1)

print("\nVQE CIRCUIT:")
print(qc)


# ============================================================
# 5. TRANSPPILE FOR IBM HARDWARE
# ============================================================

print("\nTRANSPILING CIRCUIT...")

pm=generate_preset_pass_manager(
    backend=backend,
    optimization_level=1
)

isa_circuit=pm.run(qc)

print("Transpilation complete.")


# ============================================================
# 6. MAP HAMILTONIAN TO HARDWARE LAYOUT
# ============================================================

isa_hamiltonian=hamiltonian.apply_layout(
    isa_circuit.layout
)

print("Hamiltonian mapped to hardware.")


# ============================================================
# 7. CREATE ESTIMATOR
# ============================================================

estimator=Estimator(
    mode=backend
)

estimator.options.default_shots=512


# ============================================================
# 8. HARDWARE ENERGY FUNCTION
# ============================================================

def hardware_energy(parameters):

    job=estimator.run([
        (
            isa_circuit,
            isa_hamiltonian,
            [
                parameters[0],
                parameters[1]
            ]
        )
    ])

    result=job.result()

    energy=float(
        result[0].data.evs
    )

    print(
        "theta0 =",
        round(parameters[0],4),
        "theta1 =",
        round(parameters[1],4),
        "energy =",
        round(energy,6)
    )

    return energy


# ============================================================
# 9. SPSA OPTIMIZER
# ============================================================

print("\n========================================")
print("STARTING HARDWARE VQE")
print("========================================")

optimizer=SPSA(
    maxiter=10
)


# ============================================================
# 10. RUN SPSA
# ============================================================

initial_parameters=np.array(
    [0.1,0.1]
)

result=optimizer.minimize(
    fun=hardware_energy,
    x0=initial_parameters
)


# ============================================================
# 11. FINAL RESULTS
# ============================================================

optimal_parameters=result.x

final_energy=float(result.fun)

print("\n========================================")
print("HARDWARE VQE COMPLETE")
print("========================================")

print(
    "Optimal theta0:",
    round(optimal_parameters[0],6)
)

print(
    "Optimal theta1:",
    round(optimal_parameters[1],6)
)

print(
    "Hardware VQE energy:",
    round(final_energy,6)
)

print(
    "Exact ground energy:",
    round(exact_energy,6)
)

print(
    "Absolute error:",
    round(
        abs(final_energy-exact_energy),
        6
    )
)

print("========================================")

# ============================================================
# 14. ERROR
# ============================================================

absolute_error=abs(
    result.fun-exact_energy
)

percentage_error=(
    absolute_error/
    abs(exact_energy)
)*100


print("\nAbsolute error:")
print(absolute_error)

print("\nPercentage error:")
print(percentage_error,"%")

print("\nFunction evaluations:")
print(result.nfev)

print("\nOptimization successful:")
print(result.success)


# ============================================================
# 15. CONVERGENCE GRAPH
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    range(1,len(energy_history)+1),
    energy_history,
    marker="o"
)

plt.axhline(
    exact_energy,
    linestyle="--",
    label="Exact energy"
)

plt.xlabel("VQE Iteration")
plt.ylabel("Energy")

plt.title(
    "H₂ VQE on IBM Quantum Hardware"
)

plt.legend()
plt.grid()

plt.show()